In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned data
matches = pd.read_csv('../data/processed/matches_filtered.csv')
teams_df = pd.read_csv('../data/processed/wc2026_teams.csv')
teams = teams_df['team'].tolist()

print(f"Loaded {len(matches)} matches for {len(teams)} teams")
print(f"Date range: {matches['date'].min()} to {matches['date'].max()}")
print(f"Tournaments: {matches['tournament'].unique()}")

In [ ]:
# Define K-factors by tournament type
# Every unique tournament is listed explicitly — no silent fallthrough.
# Raises ValueError for any unrecognised tournament so mismatches are caught immediately.
K_FACTOR_MAP = {
    # ── Tier 1 : FIFA World Cup final tournament (highest stakes) ────────────────
    'FIFA World Cup':                       60,
    'Confederations Cup':                   55,  # WC warm-up / prestige

    # ── Tier 2 : Continental championships (finals) ──────────────────────────────
    'UEFA Euro':                            50,
    'Copa América':                         50,
    'African Cup of Nations':               50,
    'AFC Asian Cup':                        50,
    'Gold Cup':                             50,
    'Arab Cup':                             45,
    'Gulf Cup':                             45,
    'EAFF Championship':                    45,
    'WAFF Championship':                    45,
    'COSAFA Cup':                           40,
    'CAFA Nations Cup':                     40,

    # ── Tier 3 : Nations Leagues (competitive, but not championship) ─────────────
    'UEFA Nations League':                  45,
    'CONCACAF Nations League':              45,

    # ── Tier 4 : World Cup & continental qualification ───────────────────────────
    'FIFA World Cup qualification':         40,
    'UEFA Euro qualification':              35,
    'African Cup of Nations qualification': 35,

    # ── Tier 5 : FIFA-sanctioned series / bilateral ──────────────────────────────
    'FIFA Series':                          30,
    'Superclásico de las Américas':         35,  # high-profile bilateral

    # ── Tier 6 : Invitational / regional cups ────────────────────────────────────
    'Kirin Challenge Cup':                  20,
    'Kirin Cup':                            20,
    'Al Ain International Cup':             20,
    'Canadian Shield':                      20,
    'Soccer Ashes':                         20,

    # ── Tier 7 : Friendlies ───────────────────────────────────────────────────────
    'Friendly':                             20,
}

def get_k_factor(tournament):
    """Return K-factor for *tournament*.

    Raises ValueError for any tournament not explicitly listed above,
    so data-pipeline changes are caught immediately rather than silently
    defaulting to a generic value.
    """
    if tournament not in K_FACTOR_MAP:
        raise ValueError(
            f"Unknown tournament: {tournament!r}. "
            "Add it to K_FACTOR_MAP with an appropriate K-factor."
        )
    return K_FACTOR_MAP[tournament]

# Verify every tournament in the dataset is covered (will raise if not)
print("K-factor map — all tournaments:")
for t in sorted(matches['tournament'].unique()):
    print(f"  {t:45s} {get_k_factor(t)}")


# Initialize Elo ratings
elo_ratings = {team: 1500 for team in teams}
print(f"\nInitialized {len(elo_ratings)} teams at 1500 rating")

# Sort matches by date to process chronologically
matches_sorted = matches.sort_values('date').reset_index(drop=True)

# Process each match
for idx, row in matches_sorted.iterrows():
    home_team = row['home_team']
    away_team = row['away_team']
    home_goals = row['home_score']
    away_goals = row['away_score']
    
    # Get current ratings
    home_rating = elo_ratings[home_team]
    away_rating = elo_ratings[away_team]
    
    # Calculate expected score
    expected_home = 1 / (1 + 10 ** ((away_rating - home_rating) / 400))
    expected_away = 1 - expected_home
    
    # Determine actual score (1=win, 0.5=draw, 0=loss)
    if home_goals > away_goals:
        actual_home = 1
        actual_away = 0
    elif home_goals == away_goals:
        actual_home = 0.5
        actual_away = 0.5
    else:
        actual_home = 0
        actual_away = 1
    
    # Get K-factor
    k = get_k_factor(row['tournament'])
    
    # Update ratings
    elo_ratings[home_team] = home_rating + k * (actual_home - expected_home)
    elo_ratings[away_team] = away_rating + k * (actual_away - expected_away)

print(f"\nProcessed {len(matches_sorted)} matches")
print(f"\nTop 10 teams by Elo rating:")
sorted_teams = sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)
for team, rating in sorted_teams[:10]:
    print(f"  {team:25s} {rating:7.1f}")

In [ ]:
# Save Elo ratings to CSV
elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo_rating'])
elo_df = elo_df.sort_values('elo_rating', ascending=False).reset_index(drop=True)
elo_df.to_csv('../data/processed/elo_ratings_new.csv', index=False)
print("Saved Elo ratings to data/processed/elo_ratings_new.csv")

# Summary stats
print(f"\nElo rating summary:")
print(f"  Mean: {elo_df['elo_rating'].mean():.1f}")
print(f"  Min: {elo_df['elo_rating'].min():.1f}")
print(f"  Max: {elo_df['elo_rating'].max():.1f}")